##Install Libraries

In [1]:
!pip install -q langchain
!pip install -q langchain-text-splitters
!pip install -q langchain-community
!pip install -q sentence-transformers
!pip install -q faiss-cpu
!pip install -q pypdf
!pip install -q transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 78.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 50.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 92.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 18.5 MB/s eta 0:00:00


##Import Libraries

In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from transformers import pipeline

/tmp/ipykernel_716/3176701858.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


##Load Pdf Document

In [3]:
loader = PyPDFLoader("C for beginners.pdf")
documents = loader.load()

##Splits into chunks

In [4]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
docs = text_splitter.split_documents(
    documents
)

##Create Embeddings

In [5]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/tmp/ipykernel_716/2127729888.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

##Store in Vector Database

In [6]:
vectorstore = FAISS.from_documents(
    docs,
    embedding_model
)

##Create Retriever

In [7]:
retriever = vectorstore.as_retriever()

##Load Language Model

In [8]:
#Simple HuggingFace QA model:
qa_pipeline = pipeline(
    task="text-generation",
    model="google/flan-t5-base"
)

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cohere2MoeForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DeepseekV32ForCausalLM', 'DeepseekV4ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM', 'Ernie4_5ForCausalLM', 'Ernie4_5_MoeForCausalLM', 'Exaone4ForCaus

##Ask Questions

In [9]:
query = "What is the main idea of the document?"

##Retrieve Relevant Chunks

In [10]:
retrieved_docs = retriever.invoke(query)
context = "\n".join(
    [doc.page_content for doc in retrieved_docs[:3]]
)
print(context)

• A brief introduction to pointers, how to declare and use them with example.
Proprietary content. ©Great Learning. All Rights Reserved. Unauthorized use or distribution prohibited
Introduction to C
Proprietary content. ©Great Learning. All Rights Reserved. Unauthorized use or distribution prohibited
Structures and Union


##Generate Answer

In [11]:
prompt = f"""
Answer the question based on the context below.
Context:
{context}
Question:
{query}
Answer:
"""
result = qa_pipeline(
    prompt,
    max_new_tokens=100
)
print("Question:", query)
print("Answer:", result[0]['generated_text'])

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: What is the main idea of the document?
Answer: 
Answer the question based on the context below.
Context:
• A brief introduction to pointers, how to declare and use them with example.
Proprietary content. ©Great Learning. All Rights Reserved. Unauthorized use or distribution prohibited
Introduction to C
Proprietary content. ©Great Learning. All Rights Reserved. Unauthorized use or distribution prohibited
Structures and Union
Question:
What is the main idea of the document?
Answer:



##Show Retrieved Context

In [12]:
print("Retrieved Context:\n")
print(context)

Retrieved Context:

• A brief introduction to pointers, how to declare and use them with example.
Proprietary content. ©Great Learning. All Rights Reserved. Unauthorized use or distribution prohibited
Introduction to C
Proprietary content. ©Great Learning. All Rights Reserved. Unauthorized use or distribution prohibited
Structures and Union


#Final Workflow Diagram

##RAG Workflow

PDF → Text Chunking → Embeddings → FAISS Vector Store
→ Query Embedding → Similarity Search → Retrieved Context
→ Language Model → Final Answer

## Add Multiple Questions

In [13]:
questions = [
    "What is the main topic?",
    "What skills are mentioned?",
    "What is the conclusion?"
]
for q in questions:
    # Retrieve relevant chunks
    retrieved_docs = retriever.invoke(q)
    # Extract context
    context = retrieved_docs[0].page_content
    # Create prompt
    prompt = f"""
    Answer the question based on the context below.
    Context:
    {context}
    Question:
    {q}
    Answer:
    """
    # Generate answer
    result = qa_pipeline(
        prompt,
        max_new_tokens=100
    )
    print("\nQuestion:", q)
    print("Answer:")
    print(result[0]['generated_text'])

[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Question: What is the main topic?
Answer:

    Answer the question based on the context below.
    Context:
    • A brief introduction to pointers, how to declare and use them with example.
    Question:
    What is the main topic?
    Answer:
    


[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Question: What skills are mentioned?
Answer:

    Answer the question based on the context below.
    Context:
    Proprietary content. ©Great Learning. All Rights Reserved. Unauthorized use or distribution prohibited
C for Beginners
    Question:
    What skills are mentioned?
    Answer:
    

Question: What is the conclusion?
Answer:

    Answer the question based on the context below.
    Context:
    statements are executed.
    Question:
    What is the conclusion?
    Answer:
    


## System Optimization Experiments

1. Chunk Size Experiment
- Smaller chunks improve retrieval precision.
- Larger chunks provide more context but may reduce relevance.

2. Embedding Model
- Used sentence-transformers/all-MiniLM-L6-v2
- Lightweight and effective for semantic similarity.

3. Retrieval Improvement
- Top-3 retrieved chunks were combined to improve context quality.

4. Future Improvements
- Hybrid Search (keyword + vector search)
- Re-ranking models
- Better LLMs for answer generation

##Conclusion

1. RAG combines retrieval and generation.
2. FAISS enables efficient vector similarity search.
3. Embeddings capture semantic meaning of text.
4. Retrieved context improves answer accuracy.
5. RAG systems are useful for private/custom document QA.

##Project Summary

Project: Document Question Answering System (RAG)

Dataset: PDF Document

Vector Database: FAISS

Embedding Model: all-MiniLM-L6-v2

Language Model: FLAN-T5

Task: Retrieval-Augmented Generation

## Key Learnings

- Learned how Retrieval-Augmented Generation (RAG) works internally.
- Understood semantic search using embeddings.
- Explored vector databases using FAISS.
- Built a context-aware question answering pipeline.
- Learned how retrieval improves factual accuracy in AI systems.